# Statistics: Inference and Decision Making with Calculus

## Why Statistics Matters for Calculus

Statistics heavily relies on calculus:

- **Maximum Likelihood Estimation (MLE)**: Uses derivatives to find best parameters
- **Least Squares Regression**: Minimization using calculus
- **Confidence Intervals**: Based on probability distributions (integration)
- **Hypothesis Testing**: Uses probability (calculus for continuous distributions)
- **Information Theory**: Optimization and entropy formulas

**The Big Idea**: Statistics answers "What can we learn from data?" using probability (which uses calculus) and optimization (which IS calculus)!

---

## Learning Objectives

By the end of this notebook, you will:
- Understand parameter estimation (MLE using calculus!)
- Master confidence intervals and hypothesis testing
- Apply linear regression with calculus optimization
- Connect statistical methods to calculus concepts

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.optimize import minimize
import pandas as pd

# Set style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 7)
plt.rcParams['font.size'] = 12

np.random.seed(42)

## 1. Descriptive Statistics

### Measures of Center

**Mean**: $\bar{x} = \frac{1}{n}\sum_{i=1}^n x_i$

**Median**: Middle value when sorted

**Mode**: Most frequent value

### Measures of Spread

**Variance**: $s^2 = \frac{1}{n-1}\sum_{i=1}^n (x_i - \bar{x})^2$

**Standard Deviation**: $s = \sqrt{s^2}$

**Interquartile Range (IQR)**: $Q_3 - Q_1$

### Distribution Shape

**Skewness**: Measures asymmetry

**Kurtosis**: Measures tail heaviness

In [ ]:
# Generate sample data
data_normal = np.random.normal(50, 10, 1000)
data_skewed = np.random.gamma(2, 2, 1000)

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Plot 1: Normal distribution with statistics
axes[0, 0].hist(data_normal, bins=40, density=True, alpha=0.7, color='blue', edgecolor='black')
mean_n = np.mean(data_normal)
median_n = np.median(data_normal)
std_n = np.std(data_normal, ddof=1)

axes[0, 0].axvline(mean_n, color='red', linestyle='--', linewidth=3, label=f'Mean = {mean_n:.2f}')
axes[0, 0].axvline(median_n, color='green', linestyle='--', linewidth=2, label=f'Median = {median_n:.2f}')
axes[0, 0].axvline(mean_n - std_n, color='orange', linestyle=':', linewidth=2, alpha=0.7)
axes[0, 0].axvline(mean_n + std_n, color='orange', linestyle=':', linewidth=2, alpha=0.7, label=f'±1 SD')

axes[0, 0].set_xlabel('Value', fontsize=11)
axes[0, 0].set_ylabel('Density', fontsize=11)
axes[0, 0].set_title('Symmetric Distribution (Normal)\nMean ≈ Median', fontsize=13, fontweight='bold')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Plot 2: Skewed distribution
axes[0, 1].hist(data_skewed, bins=40, density=True, alpha=0.7, color='purple', edgecolor='black')
mean_s = np.mean(data_skewed)
median_s = np.median(data_skewed)
std_s = np.std(data_skewed, ddof=1)

axes[0, 1].axvline(mean_s, color='red', linestyle='--', linewidth=3, label=f'Mean = {mean_s:.2f}')
axes[0, 1].axvline(median_s, color='green', linestyle='--', linewidth=2, label=f'Median = {median_s:.2f}')

axes[0, 1].set_xlabel('Value', fontsize=11)
axes[0, 1].set_ylabel('Density', fontsize=11)
axes[0, 1].set_title('Right-Skewed Distribution\nMean > Median', fontsize=13, fontweight='bold')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Plot 3: Box plot comparison
axes[1, 0].boxplot([data_normal, data_skewed], labels=['Normal', 'Skewed'], patch_artist=True,
                   boxprops=dict(facecolor='lightblue', alpha=0.7),
                   medianprops=dict(color='red', linewidth=2))
axes[1, 0].set_ylabel('Value', fontsize=11)
axes[1, 0].set_title('Box Plots: Visualizing Spread and Skewness', fontsize=13, fontweight='bold')
axes[1, 0].grid(True, alpha=0.3, axis='y')

# Plot 4: Summary statistics table
axes[1, 1].axis('off')
summary = f"""DESCRIPTIVE STATISTICS SUMMARY

Normal Distribution:
  Mean:     {mean_n:.3f}
  Median:   {median_n:.3f}
  Std Dev:  {std_n:.3f}
  Variance: {std_n**2:.3f}
  Min:      {np.min(data_normal):.3f}
  Max:      {np.max(data_normal):.3f}
  Q1:       {np.percentile(data_normal, 25):.3f}
  Q3:       {np.percentile(data_normal, 75):.3f}
  IQR:      {np.percentile(data_normal, 75) - np.percentile(data_normal, 25):.3f}
  Skewness: {stats.skew(data_normal):.3f}

Skewed Distribution:
  Mean:     {mean_s:.3f}
  Median:   {median_s:.3f}
  Std Dev:  {std_s:.3f}
  Variance: {std_s**2:.3f}
  Min:      {np.min(data_skewed):.3f}
  Max:      {np.max(data_skewed):.3f}
  Q1:       {np.percentile(data_skewed, 25):.3f}
  Q3:       {np.percentile(data_skewed, 75):.3f}
  IQR:      {np.percentile(data_skewed, 75) - np.percentile(data_skewed, 25):.3f}
  Skewness: {stats.skew(data_skewed):.3f} (positive = right skew)
"""
axes[1, 1].text(0.05, 0.95, summary, transform=axes[1, 1].transAxes,
               fontsize=9, verticalalignment='top', family='monospace',
               bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.3))

plt.tight_layout()
plt.show()

print("Key Insights:")
print(f"  • Symmetric distribution: Mean ≈ Median")
print(f"  • Right-skewed: Mean > Median (pulled by tail)")
print(f"  • IQR is robust to outliers (unlike standard deviation)")

## 2. Maximum Likelihood Estimation (MLE): CALCULUS!

**Goal**: Find parameter $\theta$ that maximizes the likelihood of observing our data.

### Likelihood Function

For data $x_1, ..., x_n$:
$$L(\theta) = \prod_{i=1}^n f(x_i; \theta)$$

### Log-Likelihood

Easier to work with:
$$\ell(\theta) = \log L(\theta) = \sum_{i=1}^n \log f(x_i; \theta)$$

### MLE Using Calculus!

1. Take derivative: $\frac{d\ell}{d\theta}$
2. Set equal to zero: $\frac{d\ell}{d\theta} = 0$
3. Solve for $\hat{\theta}_{MLE}$
4. Verify it's a maximum (second derivative test!)

**This is pure calculus optimization!**

In [ ]:
# MLE Example: Estimating λ for exponential distribution
print("=" * 70)
print("MAXIMUM LIKELIHOOD ESTIMATION USING CALCULUS")
print("=" * 70)

# True parameter
lambda_true = 2.0

# Generate sample data
n_samples = 100
data_exp = np.random.exponential(1/lambda_true, n_samples)

print(f"\nTrue λ = {lambda_true}")
print(f"Sample size: n = {n_samples}")
print(f"Data: {data_exp[:5].round(3)}...")

# Likelihood function: L(λ) = ∏ λe^(-λx_i) = λ^n e^(-λΣx_i)
# Log-likelihood: ℓ(λ) = n·log(λ) - λ·Σx_i

def log_likelihood(lam, data):
    n = len(data)
    return n * np.log(lam) - lam * np.sum(data)

# Derivative: dℓ/dλ = n/λ - Σx_i
# Set to 0: n/λ = Σx_i
# Solve: λ_MLE = n / Σx_i = 1 / x̄

lambda_mle = 1 / np.mean(data_exp)

print(f"\nCALCULUS SOLUTION:")
print(f"  ℓ(λ) = n·log(λ) - λ·Σx_i")
print(f"  dℓ/dλ = n/λ - Σx_i")
print(f"  Set dℓ/dλ = 0: n/λ = Σx_i")
print(f"  Solve: λ_MLE = n/Σx_i = 1/x̄")
print(f"\n  λ_MLE = 1/{np.mean(data_exp):.4f} = {lambda_mle:.4f}")
print(f"  True λ = {lambda_true}")
print(f"  Error: {abs(lambda_mle - lambda_true):.4f}")

# Visualize likelihood surface
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Plot 1: Log-likelihood function
lambda_range = np.linspace(0.5, 4, 1000)
ll_values = [log_likelihood(lam, data_exp) for lam in lambda_range]

axes[0].plot(lambda_range, ll_values, 'b-', linewidth=3, label='Log-likelihood ℓ(λ)')
axes[0].axvline(lambda_mle, color='red', linestyle='--', linewidth=2, 
               label=f'MLE: λ = {lambda_mle:.3f}')
axes[0].axvline(lambda_true, color='green', linestyle='--', linewidth=2, 
               label=f'True: λ = {lambda_true}')
axes[0].plot(lambda_mle, log_likelihood(lambda_mle, data_exp), 'ro', markersize=15)
axes[0].set_xlabel('λ', fontsize=12)
axes[0].set_ylabel('Log-likelihood ℓ(λ)', fontsize=12)
axes[0].set_title('Finding MLE by Maximizing Log-Likelihood\n(Set derivative = 0)', 
                 fontsize=13, fontweight='bold')
axes[0].grid(True, alpha=0.3)
axes[0].legend()

# Plot 2: Derivative
def log_likelihood_deriv(lam, data):
    n = len(data)
    return n/lam - np.sum(data)

deriv_values = [log_likelihood_deriv(lam, data_exp) for lam in lambda_range]

axes[1].plot(lambda_range, deriv_values, 'purple', linewidth=3, label="dℓ/dλ")
axes[1].axhline(0, color='black', linestyle='-', linewidth=1.5, label='dℓ/dλ = 0')
axes[1].axvline(lambda_mle, color='red', linestyle='--', linewidth=2, label=f'MLE: λ = {lambda_mle:.3f}')
axes[1].plot(lambda_mle, 0, 'ro', markersize=15, zorder=5)
axes[1].set_xlabel('λ', fontsize=12)
axes[1].set_ylabel('dℓ/dλ', fontsize=12)
axes[1].set_title('First Derivative: Zero at Maximum\n(Calculus Condition for Optimum)', 
                 fontsize=13, fontweight='bold')
axes[1].grid(True, alpha=0.3)
axes[1].legend()

plt.tight_layout()
plt.show()

# Second derivative test
second_deriv = -n_samples / (lambda_mle**2)
print(f"\nSECOND DERIVATIVE TEST:")
print(f"  d²ℓ/dλ² = -n/λ²")
print(f"  At λ_MLE: d²ℓ/dλ² = -{n_samples}/{lambda_mle:.3f}² = {second_deriv:.2f} < 0")
print(f"  ✓ Negative second derivative confirms MAXIMUM")

## 3. Confidence Intervals

A **confidence interval** gives a range of plausible values for a parameter.

### For a Mean (σ known)

$$\bar{x} \pm z_{\alpha/2} \frac{\sigma}{\sqrt{n}}$$

### For a Mean (σ unknown, use t-distribution)

$$\bar{x} \pm t_{\alpha/2, n-1} \frac{s}{\sqrt{n}}$$

### Interpretation

95% CI means: If we repeated this process many times, 95% of intervals would contain the true parameter.

**Calculus Connection**: The formulas come from integrating the probability distribution!

In [ ]:
# Confidence intervals demonstration
print("=" * 70)
print("CONFIDENCE INTERVALS")
print("=" * 70)

# True population parameters
mu_true = 100
sigma_true = 15

# Single sample
n = 30
sample = np.random.normal(mu_true, sigma_true, n)
xbar = np.mean(sample)
s = np.std(sample, ddof=1)

# 95% confidence interval
alpha = 0.05
t_crit = stats.t.ppf(1 - alpha/2, df=n-1)
margin_error = t_crit * s / np.sqrt(n)
ci_lower = xbar - margin_error
ci_upper = xbar + margin_error

print(f"\nSingle Sample (n={n}):")
print(f"  Sample mean: x̄ = {xbar:.2f}")
print(f"  Sample std:  s = {s:.2f}")
print(f"  t-critical (α=0.05, df={n-1}): {t_crit:.3f}")
print(f"\n  95% CI: {xbar:.2f} ± {margin_error:.2f}")
print(f"         = ({ci_lower:.2f}, {ci_upper:.2f})")
print(f"\n  True μ = {mu_true}")
print(f"  Interval {'CONTAINS' if ci_lower <= mu_true <= ci_upper else 'MISSES'} true mean")

# Simulate many CIs to show coverage
n_simulations = 100
cis = []
contains_true = []

for i in range(n_simulations):
    sample_sim = np.random.normal(mu_true, sigma_true, n)
    xbar_sim = np.mean(sample_sim)
    s_sim = np.std(sample_sim, ddof=1)
    me_sim = t_crit * s_sim / np.sqrt(n)
    ci_low = xbar_sim - me_sim
    ci_high = xbar_sim + me_sim
    cis.append((ci_low, ci_high))
    contains_true.append(ci_low <= mu_true <= ci_high)

coverage = np.mean(contains_true)

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Plot 1: Many confidence intervals
for i, (ci_low, ci_high) in enumerate(cis[:50]):  # Show first 50
    color = 'green' if contains_true[i] else 'red'
    axes[0].plot([ci_low, ci_high], [i, i], color=color, linewidth=1, alpha=0.7)
    axes[0].plot((ci_low + ci_high)/2, i, 'o', color=color, markersize=3)

axes[0].axvline(mu_true, color='blue', linestyle='--', linewidth=3, label=f'True μ = {mu_true}')
axes[0].set_xlabel('Value', fontsize=12)
axes[0].set_ylabel('Sample Number', fontsize=12)
axes[0].set_title(f'50 Different 95% Confidence Intervals\nGreen = Contains μ, Red = Misses', 
                 fontsize=13, fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3, axis='x')

# Plot 2: Distribution and CI
x_range = np.linspace(mu_true - 4*sigma_true, mu_true + 4*sigma_true, 1000)
pdf = stats.norm.pdf(x_range, mu_true, sigma_true)
axes[1].plot(x_range, pdf, 'b-', linewidth=3, label=f'Population N({mu_true}, {sigma_true}²)')

# Sampling distribution of x̄
se = sigma_true / np.sqrt(n)
pdf_xbar = stats.norm.pdf(x_range, mu_true, se)
axes[1].plot(x_range, pdf_xbar, 'r-', linewidth=3, label=f'Sampling dist of x̄')

# Show one CI
axes[1].axvline(ci_lower, color='green', linestyle='--', linewidth=2)
axes[1].axvline(ci_upper, color='green', linestyle='--', linewidth=2)
axes[1].fill_betweenx([0, max(pdf)], ci_lower, ci_upper, alpha=0.2, color='green',
                      label=f'95% CI: ({ci_lower:.1f}, {ci_upper:.1f})')
axes[1].axvline(mu_true, color='blue', linestyle='-', linewidth=2, label=f'True μ = {mu_true}')

axes[1].set_xlabel('Value', fontsize=12)
axes[1].set_ylabel('Density', fontsize=12)
axes[1].set_title('95% CI Based on Sampling Distribution', fontsize=13, fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n{n_simulations} Simulations:")
print(f"  Coverage: {sum(contains_true)}/{n_simulations} = {coverage*100:.1f}%")
print(f"  Expected: 95%")
print(f"  ✓ Empirical coverage matches theoretical prediction!")

## 4. Hypothesis Testing

### Framework

1. **Null hypothesis** $H_0$: Statement to test (usually "no effect")
2. **Alternative** $H_A$: What we suspect
3. **Test statistic**: Computed from data
4. **P-value**: Probability of seeing data this extreme if $H_0$ is true
5. **Decision**: Reject $H_0$ if p-value < $\alpha$ (usually 0.05)

### Common Tests

**One-sample t-test**:
$$t = \frac{\bar{x} - \mu_0}{s/\sqrt{n}}$$

**Two-sample t-test**:
$$t = \frac{\bar{x}_1 - \bar{x}_2}{\sqrt{\frac{s_1^2}{n_1} + \frac{s_2^2}{n_2}}}$$

**Calculus Connection**: P-values are computed by integrating probability distributions!

In [ ]:
# Hypothesis testing example
print("=" * 70)
print("HYPOTHESIS TESTING")
print("=" * 70)

# Scenario: Testing if a new teaching method improves test scores
# H0: μ = 70 (no improvement)
# HA: μ > 70 (improvement)

mu_0 = 70  # Null hypothesis mean
alpha_test = 0.05

# Sample data (new teaching method)
scores = np.array([75, 82, 78, 85, 73, 79, 88, 76, 81, 77, 84, 72, 80, 74, 83])
n_test = len(scores)
xbar_test = np.mean(scores)
s_test = np.std(scores, ddof=1)

# Test statistic
t_stat = (xbar_test - mu_0) / (s_test / np.sqrt(n_test))

# P-value (one-tailed)
df = n_test - 1
p_value = 1 - stats.t.cdf(t_stat, df)

print(f"\nScenario: New teaching method")
print(f"  H₀: μ = {mu_0} (no improvement)")
print(f"  Hₐ: μ > {mu_0} (improvement)")
print(f"  α = {alpha_test}")
print(f"\nSample Data (n={n_test}):")
print(f"  Scores: {scores}")
print(f"  x̄ = {xbar_test:.2f}")
print(f"  s = {s_test:.2f}")
print(f"\nTest Statistic:")
print(f"  t = (x̄ - μ₀)/(s/√n)")
print(f"    = ({xbar_test:.2f} - {mu_0})/{s_test:.2f}/√{n_test}")
print(f"    = {t_stat:.3f}")
print(f"\nP-value (CALCULUS: integration of t-distribution):")
print(f"  p = P(T ≥ {t_stat:.3f}) = {p_value:.4f}")
print(f"\nDecision:")
if p_value < alpha_test:
    print(f"  p-value ({p_value:.4f}) < α ({alpha_test})")
    print(f"  ✓ REJECT H₀: Significant evidence of improvement!")
else:
    print(f"  p-value ({p_value:.4f}) ≥ α ({alpha_test})")
    print(f"  ✗ FAIL TO REJECT H₀: Insufficient evidence")

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Plot 1: t-distribution with test statistic
t_range = np.linspace(-4, 6, 1000)
t_pdf = stats.t.pdf(t_range, df)

axes[0].plot(t_range, t_pdf, 'b-', linewidth=3, label=f't-distribution (df={df})')
axes[0].axvline(t_stat, color='red', linestyle='--', linewidth=3, label=f'Test statistic = {t_stat:.3f}')

# Shade p-value region
t_crit_region = t_range[t_range >= t_stat]
pdf_crit = stats.t.pdf(t_crit_region, df)
axes[0].fill_between(t_crit_region, pdf_crit, alpha=0.3, color='red', 
                     label=f'P-value = {p_value:.4f}')

# Critical value
t_critical = stats.t.ppf(1 - alpha_test, df)
axes[0].axvline(t_critical, color='green', linestyle=':', linewidth=2, 
               label=f'Critical value (α={alpha_test}): {t_critical:.3f}')

axes[0].set_xlabel('t', fontsize=12)
axes[0].set_ylabel('Density', fontsize=12)
axes[0].set_title('Hypothesis Test Visualization\nShaded Area = P-value', fontsize=13, fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Plot 2: Data distribution
axes[1].hist(scores, bins=8, density=True, alpha=0.7, color='skyblue', edgecolor='black',
            label='Observed scores')
axes[1].axvline(xbar_test, color='red', linestyle='--', linewidth=3, label=f'Sample mean = {xbar_test:.2f}')
axes[1].axvline(mu_0, color='blue', linestyle='--', linewidth=3, label=f'H₀: μ = {mu_0}')

# Overlay normal distribution
x_norm_range = np.linspace(min(scores)-5, max(scores)+5, 1000)
pdf_norm = stats.norm.pdf(x_norm_range, xbar_test, s_test)
axes[1].plot(x_norm_range, pdf_norm, 'r-', linewidth=2, alpha=0.7, label='Fitted normal')

axes[1].set_xlabel('Score', fontsize=12)
axes[1].set_ylabel('Density', fontsize=12)
axes[1].set_title('Sample Distribution', fontsize=13, fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nInterpretation:")
print(f"  The sample mean ({xbar_test:.2f}) is {xbar_test - mu_0:.2f} points higher than H₀ ({mu_0}).")
print(f"  With p-value = {p_value:.4f}, this difference is statistically significant.")
print(f"  Evidence suggests the new teaching method improves scores.")

## 5. Linear Regression: CALCULUS OPTIMIZATION!

**Goal**: Find the line $y = \beta_0 + \beta_1 x$ that best fits data.

### Least Squares: Minimize Sum of Squared Residuals

$$SSE = \sum_{i=1}^n (y_i - \hat{y}_i)^2 = \sum_{i=1}^n (y_i - \beta_0 - \beta_1 x_i)^2$$

### Using Calculus!

Take partial derivatives and set to zero:
$$\frac{\partial SSE}{\partial \beta_0} = 0, \quad \frac{\partial SSE}{\partial \beta_1} = 0$$

**Solution** (from calculus):
$$\beta_1 = \frac{\sum(x_i - \bar{x})(y_i - \bar{y})}{\sum(x_i - \bar{x})^2}$$
$$\beta_0 = \bar{y} - \beta_1\bar{x}$$

**This is pure calculus optimization - just like finding critical points!**

In [ ]:
# Linear regression with calculus
print("=" * 70)
print("LINEAR REGRESSION: LEAST SQUARES VIA CALCULUS")
print("=" * 70)

# Generate data with linear relationship + noise
np.random.seed(42)
x_reg = np.linspace(0, 10, 50)
beta0_true = 5
beta1_true = 2
y_reg = beta0_true + beta1_true * x_reg + np.random.normal(0, 2, 50)

# Compute least squares estimates using calculus formulas
x_mean = np.mean(x_reg)
y_mean = np.mean(y_reg)
beta1_hat = np.sum((x_reg - x_mean) * (y_reg - y_mean)) / np.sum((x_reg - x_mean)**2)
beta0_hat = y_mean - beta1_hat * x_mean

# Predictions
y_pred = beta0_hat + beta1_hat * x_reg
residuals = y_reg - y_pred
sse = np.sum(residuals**2)

# R-squared
sst = np.sum((y_reg - y_mean)**2)
r_squared = 1 - sse/sst

print(f"\nData: y = {beta0_true} + {beta1_true}x + noise")
print(f"\nCALCULUS SOLUTION (Least Squares):")
print(f"  ∂SSE/∂β₀ = 0 and ∂SSE/∂β₁ = 0 gives:")
print(f"\n  β₁ = Σ(xᵢ-x̄)(yᵢ-ȳ) / Σ(xᵢ-x̄)²")
print(f"     = {np.sum((x_reg - x_mean) * (y_reg - y_mean)):.2f} / {np.sum((x_reg - x_mean)**2):.2f}")
print(f"     = {beta1_hat:.4f}")
print(f"\n  β₀ = ȳ - β₁x̄")
print(f"     = {y_mean:.4f} - {beta1_hat:.4f}·{x_mean:.4f}")
print(f"     = {beta0_hat:.4f}")
print(f"\nEstimated model: ŷ = {beta0_hat:.3f} + {beta1_hat:.3f}x")
print(f"True model:      y = {beta0_true} + {beta1_true}x")
print(f"\nGoodness of fit:")
print(f"  SSE (sum squared errors) = {sse:.2f}")
print(f"  R² = {r_squared:.4f} ({r_squared*100:.1f}% variance explained)")

# Visualize
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Plot 1: Data and fitted line
axes[0, 0].scatter(x_reg, y_reg, alpha=0.6, s=50, label='Data')
axes[0, 0].plot(x_reg, y_pred, 'r-', linewidth=3, label=f'ŷ = {beta0_hat:.2f} + {beta1_hat:.2f}x')
axes[0, 0].plot(x_reg, beta0_true + beta1_true * x_reg, 'g--', linewidth=2, 
               label=f'True: y = {beta0_true} + {beta1_true}x', alpha=0.7)

# Show residuals for a few points
for i in [5, 15, 25, 35, 45]:
    axes[0, 0].plot([x_reg[i], x_reg[i]], [y_reg[i], y_pred[i]], 'k--', linewidth=1, alpha=0.5)

axes[0, 0].set_xlabel('x', fontsize=12)
axes[0, 0].set_ylabel('y', fontsize=12)
axes[0, 0].set_title(f'Linear Regression\nR² = {r_squared:.4f}', fontsize=13, fontweight='bold')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Plot 2: Residual plot
axes[0, 1].scatter(y_pred, residuals, alpha=0.6, s=50)
axes[0, 1].axhline(0, color='red', linestyle='--', linewidth=2)
axes[0, 1].set_xlabel('Fitted values', fontsize=12)
axes[0, 1].set_ylabel('Residuals', fontsize=12)
axes[0, 1].set_title('Residual Plot\n(Should be random around 0)', fontsize=13, fontweight='bold')
axes[0, 1].grid(True, alpha=0.3)

# Plot 3: SSE surface (for visualization)
beta0_range = np.linspace(beta0_hat - 3, beta0_hat + 3, 50)
beta1_range = np.linspace(beta1_hat - 0.5, beta1_hat + 0.5, 50)
B0, B1 = np.meshgrid(beta0_range, beta1_range)
SSE_surface = np.zeros_like(B0)

for i in range(len(beta0_range)):
    for j in range(len(beta1_range)):
        y_temp = B0[j, i] + B1[j, i] * x_reg
        SSE_surface[j, i] = np.sum((y_reg - y_temp)**2)

contour = axes[1, 0].contour(B0, B1, SSE_surface, levels=20, cmap='viridis')
axes[1, 0].plot(beta0_hat, beta1_hat, 'r*', markersize=20, label='Minimum (least squares)')
axes[1, 0].set_xlabel('β₀ (intercept)', fontsize=12)
axes[1, 0].set_ylabel('β₁ (slope)', fontsize=12)
axes[1, 0].set_title('SSE Surface\n(Minimized by Calculus)', fontsize=13, fontweight='bold')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)
plt.colorbar(contour, ax=axes[1, 0], label='SSE')

# Plot 4: Derivation summary
axes[1, 1].axis('off')
derivation = f"""CALCULUS DERIVATION

Minimize: SSE = Σ(yᵢ - β₀ - β₁xᵢ)²

Take partial derivatives:
  ∂SSE/∂β₀ = -2Σ(yᵢ - β₀ - β₁xᵢ)
  ∂SSE/∂β₁ = -2Σxᵢ(yᵢ - β₀ - β₁xᵢ)

Set equal to zero:
  Σ(yᵢ - β₀ - β₁xᵢ) = 0
  Σxᵢ(yᵢ - β₀ - β₁xᵢ) = 0

Solve simultaneously:
  From first equation:
    nβ₀ + β₁Σxᵢ = Σyᵢ
    β₀ = ȳ - β₁x̄

  Substitute into second:
    β₁ = Σ(xᵢ-x̄)(yᵢ-ȳ) / Σ(xᵢ-x̄)²

Result:
  β₀ = {beta0_hat:.4f}
  β₁ = {beta1_hat:.4f}

This minimizes SSE = {sse:.2f}
"""
axes[1, 1].text(0.05, 0.95, derivation, transform=axes[1, 1].transAxes,
               fontsize=9, verticalalignment='top', family='monospace',
               bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.3))

plt.tight_layout()
plt.show()

print(f"\nSecond Derivative Test (confirm minimum):")
print(f"  ∂²SSE/∂β₀² > 0 ✓")
print(f"  ∂²SSE/∂β₁² > 0 ✓")
print(f"  Hessian is positive definite ⟹ MINIMUM")

## 6. Key Takeaways

### Estimation
- **MLE**: Find parameters by **maximizing likelihood using calculus**
  - Take derivative, set to zero, solve
  - Second derivative test confirms maximum

### Inference
- **Confidence Intervals**: Range of plausible parameter values
  - Based on sampling distributions (integration!)
- **Hypothesis Testing**: Evidence against null hypothesis
  - P-values computed by **integrating** probability distributions

### Regression
- **Least Squares**: Minimize SSE using **calculus optimization**
  - ∂SSE/∂β₀ = 0 and ∂SSE/∂β₁ = 0
  - Closed-form solution from partial derivatives
- **R²**: Proportion of variance explained

### CALCULUS IS EVERYWHERE IN STATISTICS!
- **Integration**: Probabilities, CDFs, expected values
- **Differentiation**: MLE, finding optima, regression
- **Optimization**: Least squares, maximum likelihood

---

**Next Up**: Simulation - using numerical methods when calculus gets too hard!